### Simple Water balance calculation

In [38]:
import pandas as pd
import numpy as np

In [39]:
def water_balance(avg_hs, avg_rp, avg_dis, area, cell, ofstorage, label_discharge="dis_0"):
    """All output units are in mm"""
    P = avg_hs["pre_0"]
    AET = avg_hs["aet_0"]
    R = avg_hs["rch_0"]  
    OF = avg_hs["run_0"]
    Qgw = avg_hs["gdh_0"]*1000.0
    TL = avg_hs["tls_0"]*1000/np.power(cell,2)
    Peff = avg_hs["pre_0"] - avg_hs["inf_0"]
    Qgw = avg_hs["gdh_0"]*1000.
    Q = avg_dis[label_discharge]*1000/area    
    factor = avg_hs["tls_0"]/avg_rp["tls_0"]
    AER = avg_rp["aet_0"]*factor
    Rfch = avg_rp["fch_0"]*factor
    Rdch = avg_hs["rch_0"] - Rfch
    ofstorage = ofstorage*factor*1000/np.power(cell,2)
    Qbc = avg_hs["chb_0"]*1000.
    EGW = avg_hs["egw_0"]
    MBuz = avg_hs["inf_0"] - avg_hs["aet_0"] - Rdch
    MB = (avg_hs["pre_0"] - avg_hs["aet_0"] - avg_hs["egw_0"]
        - Q - AER - ofstorage - avg_hs["twsc_0"] - Qbc)
    
    # store fluxes in a dictionary
    fluxes = {"P": [P, 100.0],
              "AET": [AET, AET*100./P], "AER": [AER, AER*100./P],  "EGW": [EGW, EGW*100./P],
              "R": [R, R*100./P], "Rfch": [Rfch, Rfch*100./P], "Rdch": [Rdch, Rdch*100./P],
              "Q": [Q, Q*100./P], "Qgw": [Qgw, Qgw*100./P],
              "Peff": [Peff, Peff*100./P],
              "OF": [OF, OF*100./P], "TL": [TL, TL*100./P],              
              "TSWS":[ofstorage, ofstorage*100./P],
              "TWSA":[avg_hs["twsc_0"],avg_hs["twsc_0"]*100./P],
              "Qbc": [Qbc, Qbc*100./P],
              "MB": [MB, MB*100./P],
             "MBuz": [MBuz, MBuz*100./P]}
    return pd.DataFrame.from_dict(fluxes)

def calculate_water_balance_from_csv(fhillslope, friparian, fdischarge, area, cell, label_discharge="dis_0"):
    avg_hs = pd.read_csv(fhillslope).sum()
    avg_rp = pd.read_csv(friparian).sum()
    avg_dis = pd.read_csv(fdischarge).sum()
    avg_rp_mean = pd.read_csv(friparian).mean()
    ofstorage = avg_rp_mean["ssz_0"]
    return water_balance(avg_hs, avg_rp, avg_dis, area, cell, ofstorage, label_discharge)

#### Tilted-V basin test

In [40]:
folder = "/home/andresqm/Documents/GitHub/DRYPv2.0.1/tests/tilted_v/output/"
fname_avg_hs = folder + "test_avg.csv"
fname_avg_rp = folder + "test_RZ_avg.csv"
fname_avg_dis = folder + "test_p_dis.csv"
cell_size = 1000 # m
basin_area = (cell_size**2)*7*10 # m2

In [41]:
calculate_water_balance_from_csv(fname_avg_hs, fname_avg_rp, fname_avg_dis,
                                 basin_area, cell_size)

/tmp/ipykernel_7306/1726732392.py:40: FutureWarning: The default value of numeric_only in DataFrame.mean is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  avg_rp_mean = pd.read_csv(friparian).mean()


,P,AET,AER,EGW,R,Rfch,Rdch,Q,Qgw,Peff,OF,TL,TSWS,TWSA,Qbc,MB,MBuz
0,1499.75,815.386134,0.000154,63.667404,548.820619,0.0,548.820619,278.084841,199.511999,0.0,278.084841,6.774715e-19,0.0,341.828427,0.0,0.783041,135.543247
1,100.00,54.368137,0.000010,4.245201,36.594140,0.0,36.594140,18.542080,13.303017,0.0,18.542080,4.517229e-20,0.0,22.792361,0.0,0.052211,9.037723
